# 01 — Build NRC Lexicon Lookup Dicts

Loads three NRC lexicons, builds word-level lookup dicts, and pickles them to `data/lexicons/`
so the scoring notebook doesn't have to reload and rejoin on every run.

Run this once (or whenever you update the lexicon files).

In [1]:
import pickle
import re
from pathlib import Path

import nltk
import pandas as pd
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

LEXICON_DIR = Path('data/lexicons')
LEXICON_DIR.mkdir(parents=True, exist_ok=True)

EMOTIONS = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']
VAD_DIMS  = ['valence', 'arousal', 'dominance']

_lemmatizer = WordNetLemmatizer()


def _tokenize(keyword: str) -> list:
    """Split phrase on whitespace/hyphens/underscores, lowercase, alpha tokens only."""
    return [t for t in re.split(r'[\s\-_/]+', keyword.lower()) if t.isalpha()]


def _lemma_candidates(tok: str) -> list:
    """Return lemma forms for noun, verb, adjective POS (deduplicated)."""
    seen, out = set(), []
    for pos in ('n', 'v', 'a'):
        lemma = _lemmatizer.lemmatize(tok, pos=pos)
        if lemma not in seen:
            seen.add(lemma)
            out.append(lemma)
    if tok not in seen:
        out.append(tok)
    return out

In [2]:
# ── NRC EmoLex (binary 0/1 per emotion + positive/negative) ──────────────
NRC_EMOLEX = Path('data/NRC-emotion-lexicon-wordlevel-alphabetized-v0.92.txt')
nrc_long = pd.read_csv(NRC_EMOLEX, sep='\t', header=None,
                       names=['word', 'emotion', 'value'], skiprows=46)
nrc_wide = nrc_long.pivot(index='word', columns='emotion', values='value').reset_index()
print(f'EmoLex: {len(nrc_wide):,} words')

EmoLex: 14,150 words


In [3]:
# ── NRC Emotion Intensity (continuous 0–1 per emotion) ───────────────────
INTENSITY_FILE = Path('data/NRC-Emotion-Intensity-Lexicon/NRC-Emotion-Intensity-Lexicon-v1.txt')
nrc_int_long = pd.read_csv(INTENSITY_FILE, sep='\t', header=None,
                           names=['word', 'emotion', 'intensity'])
nrc_int_wide = (
    nrc_int_long
    .pivot(index='word', columns='emotion', values='intensity')
    .fillna(0)
    .reset_index()
)
print(f'Intensity lexicon: {len(nrc_int_wide):,} words')

Intensity lexicon: 5,891 words


In [4]:
# ── NRC VAD v2.1 (bipolar scale -1 to +1) ───────────────────────────────
# MWE (multi-word expressions) checked first — e.g. "child abuse" scored
# directly rather than averaging "child" + "abuse" individually.
# Falls back to unigram lookup for single tokens / unmatched phrases.

VAD_UNIGRAM = Path('data/NRC-VAD-Lexicon-v2.1/Unigrams/unigrams-NRC-VAD-Lexicon-v2.1.txt')
VAD_MWE     = Path('data/NRC-VAD-Lexicon-v2.1/MWE/mwe-NRC-VAD-Lexicon-v2.1.txt')

vad_unigram = pd.read_csv(VAD_UNIGRAM, sep='\t')
vad_mwe     = pd.read_csv(VAD_MWE, sep='\t')

print(f'VAD unigrams: {len(vad_unigram):,} words')
print(f'VAD MWEs:     {len(vad_mwe):,} expressions')
print(f'Scale:        bipolar (-1 to +1,  0 = neutral)')

# Build lookup dicts: term → {valence, arousal, dominance}
vad_unigram_lookup = {
    row['term']: {d: row[d] for d in VAD_DIMS}
    for _, row in vad_unigram.iterrows()
}
vad_mwe_lookup = {
    row['term']: {d: row[d] for d in VAD_DIMS}
    for _, row in vad_mwe.iterrows()
}

# Spot check
for word in ['child abuse', 'death', 'love', 'joy']:
    hit = vad_mwe_lookup.get(word) or vad_unigram_lookup.get(word)
    src = 'MWE' if word in vad_mwe_lookup else 'unigram'
    print(f'  {word:20s} ({src}): valence={hit["valence"]:.3f}' if hit else f'  {word}: not found')

VAD unigrams: 44,728 words
VAD MWEs:     10,073 expressions
Scale:        bipolar (-1 to +1,  0 = neutral)


  child abuse          (MWE): valence=-0.566
  death                (unigram): valence=-0.938
  love                 (unigram): valence=0.996
  joy                  (unigram): valence=0.960


In [5]:
# ── Build fast lookup dicts ───────────────────────────────────────────────
intensity_lookup = {
    row['word']: {e: row[e] for e in EMOTIONS}
    for _, row in nrc_int_wide.iterrows()
}

# vad_lookup: MWE takes priority over unigram
# Scoring logic in 02_score_and_export checks MWE first (full phrase),
# then falls back to per-token unigram lookup with lemmatization.
vad_lookup = {**vad_unigram_lookup, **vad_mwe_lookup}  # MWE overwrites unigram for same key

print(f'intensity_lookup: {len(intensity_lookup):,} entries')
print(f'vad_lookup:       {len(vad_lookup):,} entries (unigram + MWE combined)')
print(f'  of which MWE:   {len(vad_mwe_lookup):,} expressions')

intensity_lookup: 5,891 entries
vad_lookup:       54,801 entries (unigram + MWE combined)
  of which MWE:   10,073 expressions


In [6]:
# ── Pickle everything needed by 02_score_and_export ──────────────────────
payload = {
    'intensity_lookup':  intensity_lookup,
    'vad_lookup':        vad_lookup,       # unigram + MWE combined
    'vad_mwe_lookup':    vad_mwe_lookup,   # MWE-only for phrase-first lookup
    'EMOTIONS':          EMOTIONS,
    'VAD_DIMS':          VAD_DIMS,
}

out = LEXICON_DIR / 'nrc_lookups.pkl'
with open(out, 'wb') as f:
    pickle.dump(payload, f, protocol=5)

print(f'Saved → {out}  ({out.stat().st_size / 1e6:.1f} MB)')

Saved → data/lexicons/nrc_lookups.pkl  (3.9 MB)
